# 03. Curve Calibration and LGM Parameter Extraction

This notebook connects two things that are often shown separately: the fitted market curves and the LGM parameter
payload. The point is to keep the model story anchored to the same ORE artifacts that produced the market view.

**Purpose**
- show how the calibration instruments are discovered from ORE inputs
- visualize the fitted discount curves in a way that exposes shape, density, and pillar coverage
- connect those results to the extracted LGM alpha and kappa term structures

**What you will learn**
- how the repo traces discount and index curves from a known case
- how to inspect fitted curves beyond one flat table
- where the LGM parameter term structures come from


In [ ]:
from pathlib import Path
import os
import sys

def _pythonorerunner_root(candidate: Path) -> bool:
    return (
        (candidate / "notebook_series" / "series_helpers.py").exists()
        and ((candidate / "pythonore").exists() or (candidate / "src" / "pythonore").exists())
    )

def _engine_root(candidate: Path) -> bool:
    return (candidate / "Tools" / "PythonOreRunner" / "notebook_series" / "series_helpers.py").exists()

def _find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if _pythonorerunner_root(candidate) or _engine_root(candidate):
            return candidate
    repo_hint = Path("/Users/gordonlee/Documents/Engine")
    if _engine_root(repo_hint):
        return repo_hint
    standalone_hint = Path("/Users/gordonlee/Documents/PythonOreRunner")
    if _pythonorerunner_root(standalone_hint):
        return standalone_hint
    raise RuntimeError("Could not locate the notebook repo root from the current notebook working directory")

REPO_ROOT = _find_repo_root(Path.cwd())
NOTEBOOK_DIR = REPO_ROOT / "notebook_series" if _pythonorerunner_root(REPO_ROOT) else REPO_ROOT / "Tools" / "PythonOreRunner" / "notebook_series"
for path in (NOTEBOOK_DIR, REPO_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/codex-mplconfig")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import series_helpers as nh

try:
    from IPython.display import display
except Exception:
    def display(obj):
        if hasattr(obj, "to_string"):
            print(obj.to_string())
        else:
            print(obj)

repo = nh.bootstrap_notebook_env(REPO_ROOT)
nh.apply_plot_style()
print(repo)


In [ ]:
import json
import subprocess

from py_ore_tools import (
    discount_factors_to_dataframe,
    extract_discount_factors_by_currency,
    extract_market_instruments_by_currency,
    extract_market_instruments_by_currency_from_quotes,
    fit_discount_curves_from_ore_market,
    fit_discount_curves_from_programmatic_quotes,
    fitted_curves_to_dataframe,
    parse_lgm_params_from_calibration_xml,
    parse_lgm_params_from_simulation_xml,
    quote_dicts_from_pairs,
)
from ore_curve_fit_parity import compare_python_vs_ore, trace_discount_curve_from_ore, trace_index_curve_from_ore

ORE_XML = nh.default_curve_case_ore_xml(repo)
CALIBRATION_PARITY_SCRIPT = REPO_ROOT / "src" / "pythonore" / "benchmarks" / "benchmark_lgm_calibration_parity.py"
df_payload = extract_discount_factors_by_currency(ORE_XML, configuration_id="default")
df_long = discount_factors_to_dataframe(df_payload)
instruments = extract_market_instruments_by_currency(ORE_XML)
fitted = fit_discount_curves_from_ore_market(
    ORE_XML,
    fit_method="bootstrap_mm_irs_v1",
    fit_grid_mode="dense",
    dense_step_years=0.25,
)
fitted_df = fitted_curves_to_dataframe(fitted)
usd_discount_trace = trace_discount_curve_from_ore(ORE_XML, currency="USD")
usd_index_trace = trace_index_curve_from_ore(ORE_XML, index_name="USD-LIBOR-6M")
usd_python_fit_compare = compare_python_vs_ore(ORE_XML, currency="USD")
configured_lgm, calibrated_lgm, calibration_meta = nh.run_fresh_lgm_calibration_demo(ccy_key="EUR")
calibration_hw_parity_meta = nh.run_fresh_lgm_calibration_hullwhite_parity_demo(
    base_fresh_ore_xml=calibration_meta["fresh_ore_xml"]
)

print("ORE XML:", ORE_XML)
display(df_long.head(12))
display(pd.DataFrame.from_dict(instruments, orient="index").reset_index(names="currency"))


## Inputs we reuse from the repo

The curve side of the series is grounded in the existing parity walkthroughs and tests:
- `ore_curve_fit_parity/notebooks/usd_curve_parity_walkthrough.ipynb`
- `ore_curve_fit_parity/notebooks/multi_currency_curve_trace_demo.ipynb`
- `tests/test_quick_curve_fit.py`


In [ ]:
# Start with the size of each fitted grid and the number of instruments that drove it.
summary = (
    fitted_df.groupby("ccy", as_index=False)
    .agg(
        fit_points=("time", "count"),
        first_df=("df", "first"),
        last_df=("df", "last"),
        instrument_count=("instrument_count", "max"),
    )
)
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
nh.plot_bar_frame(summary, "ccy", "fit_points", title="Fit grid size by currency", color=nh.PALETTE["blue"], ax=axes[0])
nh.plot_bar_frame(summary, "ccy", "instrument_count", title="Instrument count by currency", color=nh.PALETTE["teal"], ax=axes[1])
plt.tight_layout()
plt.show()
plt.close(fig)


The first chart answers a structural question: how dense is the fitted curve relative to the number of calibration
instruments? That difference is exactly why a fitted grid can look smooth even when the market input is sparse.


In [ ]:
nh.plot_log_discount_factors(df_long, title="Extracted log(DF) by currency")
heatmap_df = nh.zero_rate_heatmap_frame(df_long)
display(heatmap_df.head(12))
nh.plot_zero_rate_heatmap(heatmap_df, title="Extracted zero-rate heatmap by currency and maturity")

extractor_audit = (
    df_long.groupby("ccy", as_index=False)
    .agg(
        point_count=("time", "count"),
        min_time=("time", "min"),
        max_time=("time", "max"),
        min_df=("df", "min"),
        max_df=("df", "max"),
    )
    .sort_values("ccy")
)
display(extractor_audit)


The older extractor notebook also tried to expose this through a session bridge. In the current codebase that bridge
is not surfaced as a public `XVASession` method, so this series keeps the extraction path direct and explicit rather
than pretending the runtime API already exposes a stable curve-extraction wrapper.


In [ ]:
nh.plot_fitted_curves({ccy: fitted[ccy] for ccy in sorted(fitted)[:4]}, title="Representative fitted discount curves")
nh.plot_curve_diagnostics(fitted_df, ccy="USD", title="USD curve diagnostics from the fitted output grid")

trace_compare = pd.DataFrame(
    {
        "series": ["USD discount native nodes", "USD discount report grid", "USD 6M dependency graph"],
        "count": [
            len(usd_discount_trace["ore_calibration_trace"]["pillars"]),
            len(usd_discount_trace["ore_curve_points"]["times"]),
            len(usd_index_trace["dependency_graph"]),
        ],
    }
)
display(trace_compare)


## Python curve fitter demo on mixed instruments

The USD discount trace is a good compact demo because it mixes three instrument types in one curve:

- deposits
- FRAs
- swaps

The Python diagnostic fitter here is deliberately narrow: it rebuilds the ORE curve from the native ORE
calibration nodes and then compares the rebuilt discount factors to the ORE report grid. That makes the
interpolation parity visible without pretending we have fully reimplemented ORE helper construction.


In [ ]:
usd_input_rows = []
for segment in usd_discount_trace["segment_alignment"]:
    for quote in segment["quotes"]:
        pillar = quote.get("ore_pillar") or {}
        usd_input_rows.append(
            {
                "segment_type": segment["type"],
                "quote_key": quote["quote_key"],
                "market_quote": quote.get("quote_value"),
                "pillar_time": pd.to_numeric(pillar.get("time"), errors="coerce"),
                "pillar_df": pd.to_numeric(pillar.get("discountFactor"), errors="coerce"),
                "pillar_zero_rate": pd.to_numeric(pillar.get("zeroRate"), errors="coerce"),
            }
        )
usd_input_df = pd.DataFrame(usd_input_rows)
display(usd_input_df.head(12))

usd_fit_compare_df = pd.DataFrame(
    [
        {
            "time": point.time,
            "ore_df": point.ore_value,
            "python_df": point.engine_value,
            "abs_error": point.abs_error,
            "rel_error": point.rel_error,
        }
        for point in usd_python_fit_compare.points
    ]
)
usd_fit_compare_df["ore_zero_rate"] = -np.log(np.clip(usd_fit_compare_df["ore_df"], 1.0e-12, None)) / usd_fit_compare_df["time"].replace(0.0, np.nan)
usd_fit_compare_df["python_zero_rate"] = -np.log(np.clip(usd_fit_compare_df["python_df"], 1.0e-12, None)) / usd_fit_compare_df["time"].replace(0.0, np.nan)
ore_log_df = -np.log(np.clip(usd_fit_compare_df["ore_df"].to_numpy(dtype=float), 1.0e-12, None))
py_log_df = -np.log(np.clip(usd_fit_compare_df["python_df"].to_numpy(dtype=float), 1.0e-12, None))
grid_time = usd_fit_compare_df["time"].to_numpy(dtype=float)
usd_fit_compare_df["ore_forward_rate"] = np.gradient(ore_log_df, grid_time, edge_order=1)
usd_fit_compare_df["python_forward_rate"] = np.gradient(py_log_df, grid_time, edge_order=1)
display(usd_fit_compare_df.head(12))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

native_nodes = pd.DataFrame(
    {
        "time": usd_discount_trace["native_curve_nodes"]["times"],
        "df": usd_discount_trace["native_curve_nodes"]["discount_factors"],
    }
)
axes[0].scatter(native_nodes["time"], native_nodes["df"], s=28, color=nh.PALETTE["gold"], label="ORE native nodes")
axes[0].plot(usd_fit_compare_df["time"], usd_fit_compare_df["ore_df"], linewidth=1.8, label="ORE report grid")
axes[0].plot(usd_fit_compare_df["time"], usd_fit_compare_df["python_df"], linewidth=1.4, linestyle="--", label="Python fitter")
axes[0].set_title("USD curve fitter: inputs to fitted outputs")
axes[0].set_xlabel("Time (years)")
axes[0].set_ylabel("Discount Factor")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(usd_fit_compare_df["time"], 1.0e9 * usd_fit_compare_df["abs_error"], linewidth=1.8, color=nh.PALETTE["rose"])
axes[1].set_title("USD curve fitter absolute error (nanodf)")
axes[1].set_xlabel("Time (years)")
axes[1].set_ylabel("Abs Error x 1e9")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
plt.close(fig)

rate_display = usd_fit_compare_df[
    [
        "time",
        "ore_zero_rate",
        "python_zero_rate",
        "ore_forward_rate",
        "python_forward_rate",
        "abs_error",
    ]
].head(12)
display(rate_display)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharex=True)
axes[0].plot(usd_fit_compare_df["time"], usd_fit_compare_df["ore_zero_rate"], linewidth=1.8, label="ORE zero")
axes[0].plot(usd_fit_compare_df["time"], usd_fit_compare_df["python_zero_rate"], linewidth=1.4, linestyle="--", label="Python zero")
axes[0].set_title("USD curve fitter zero rates")
axes[0].set_xlabel("Time (years)")
axes[0].set_ylabel("Zero Rate")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(usd_fit_compare_df["time"], usd_fit_compare_df["ore_forward_rate"], linewidth=1.8, label="ORE forward")
axes[1].plot(usd_fit_compare_df["time"], usd_fit_compare_df["python_forward_rate"], linewidth=1.4, linestyle="--", label="Python forward")
axes[1].set_title("USD curve fitter instantaneous forwards")
axes[1].set_xlabel("Time (years)")
axes[1].set_ylabel("Forward Rate")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()
plt.close(fig)

print("Python fitter status:", usd_python_fit_compare.status)
print("Max abs error:", usd_python_fit_compare.max_abs_error)
print("Max rel error:", usd_python_fit_compare.max_rel_error)


In [ ]:
fitted_weighted_dense = fit_discount_curves_from_ore_market(
    ORE_XML,
    fit_method="weighted_zero_logdf_v1",
    fit_grid_mode="dense",
    dense_step_years=0.25,
)
fitted_weighted_dense_df = fitted_curves_to_dataframe(fitted_weighted_dense)

sample_ccys = sorted(fitted.keys())[:4]
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ccy in sample_ccys:
    dense = fitted_weighted_dense_df[fitted_weighted_dense_df["ccy"] == ccy].sort_values("time")
    axes[0].plot(dense["time"], dense["zero_rate"], linewidth=1.8, label=ccy)
axes[0].set_title("Weighted zero fit")
axes[0].set_xlabel("Time (years)")
axes[0].set_ylabel("Zero Rate")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

for ccy in sample_ccys:
    dense = fitted_df[fitted_df["ccy"] == ccy].sort_values("time")
    axes[1].plot(dense["time"], dense["zero_rate"], linewidth=1.8, label=ccy)
axes[1].set_title("Bootstrap fit")
axes[1].set_xlabel("Time (years)")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()
plt.close(fig)


The weighted-versus-bootstrap comparison is worth showing because it makes the fitting choice visible. The old
extractor demo did this well, and it is more informative than showing one fitter in isolation.


In [ ]:
programmatic_quote_pairs = [
    ("MM/RATE/USD/USD-LIBOR/1M", 0.0520),
    ("MM/RATE/USD/USD-LIBOR/3M", 0.0515),
    ("ZERO/RATE/USD/1Y", 0.0500),
    ("IR_SWAP/RATE/USD/USD-LIBOR-3M/3M/2Y", 0.0485),
    ("IR_SWAP/RATE/USD/USD-LIBOR-3M/3M/5Y", 0.0470),
    ("MM/RATE/EUR/EUR-EURIBOR/1M", 0.0310),
    ("ZERO/RATE/EUR/1Y", 0.0300),
    ("IR_SWAP/RATE/EUR/EUR-EURIBOR-6M/6M/2Y", 0.0310),
    ("IR_SWAP/RATE/EUR/EUR-EURIBOR-6M/6M/5Y", 0.0320),
]
programmatic_quotes = quote_dicts_from_pairs(programmatic_quote_pairs)
prog_inst = extract_market_instruments_by_currency_from_quotes(asof_date="2026-03-08", quotes=programmatic_quotes)
prog_fitted = fit_discount_curves_from_programmatic_quotes(
    asof_date="2026-03-08",
    quotes=programmatic_quotes,
    fit_method="bootstrap_mm_irs_v1",
    fit_grid_mode="dense",
    dense_step_years=0.25,
)
prog_fitted_df = fitted_curves_to_dataframe(prog_fitted)

display(pd.DataFrame(
    [
        {"ccy": ccy, "instrument_count": p["instrument_count"], "fit_points_count": prog_fitted[ccy]["fit_points_count"]}
        for ccy, p in sorted(prog_inst.items())
    ]
))

fig, ax = plt.subplots(figsize=(10, 5))
for ccy, grp in prog_fitted_df.groupby("ccy"):
    g = grp.sort_values("time")
    ax.plot(g["time"], g["zero_rate"], linewidth=1.8, label=f"{ccy} bootstrap")
ax.set_title("Programmatic fitted zero curves (no XML)")
ax.set_xlabel("Time (years)")
ax.set_ylabel("Zero Rate")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()
plt.close(fig)


## Fresh LGM calibration run

The curve extraction above used a compact benchmark case because it is convenient for inspecting discount curves.
For the model section we switch to a calibration-enabled ORE example and run it fresh inside the notebook.

That gives us two distinct objects to compare:
- the LGM parameter template from `simulation_lgm.xml`
- the calibrated term structure written by ORE to `calibration.xml`

In this example ORE calibrates `alpha` and keeps `kappa` fixed, so the interesting movement is in the volatility
buckets rather than the reversion line.


In [ ]:
calibration_status = pd.DataFrame(
    [
        {"field": "source_ore_xml", "value": calibration_meta["source_ore_xml"]},
        {"field": "fresh_ore_xml", "value": calibration_meta["fresh_ore_xml"]},
        {"field": "simulation_xml", "value": calibration_meta["simulation_xml"]},
        {"field": "calibration_xml", "value": calibration_meta["calibration_xml"]},
        {"field": "ore_elapsed_sec", "value": round(calibration_meta["elapsed_sec"], 3)},
        {"field": "calibrate_vol", "value": calibration_meta["calibrate_vol"]},
        {"field": "calibrate_kappa", "value": calibration_meta["calibrate_kappa"]},
    ]
)
display(calibration_status)

alpha_configured = nh.lgm_param_frame(configured_lgm, param="alpha").rename(columns={"value": "configured_alpha"})
alpha_calibrated = nh.lgm_param_frame(calibrated_lgm, param="alpha").rename(columns={"value": "calibrated_alpha"})
kappa_configured = nh.lgm_param_frame(configured_lgm, param="kappa").rename(columns={"value": "configured_kappa"})
kappa_calibrated = nh.lgm_param_frame(calibrated_lgm, param="kappa").rename(columns={"value": "calibrated_kappa"})

display(alpha_configured)
display(alpha_calibrated)
display(kappa_configured)
display(kappa_calibrated)

alpha_compare = alpha_calibrated.merge(
    alpha_configured[["bucket_index", "configured_alpha"]],
    on="bucket_index",
    how="left",
)
alpha_compare["configured_alpha"] = alpha_compare["configured_alpha"].fillna(alpha_configured["configured_alpha"].iloc[-1])
alpha_compare["alpha_shift_bp"] = 1.0e4 * (alpha_compare["calibrated_alpha"] - alpha_compare["configured_alpha"])
display(alpha_compare[["bucket_label", "configured_alpha", "calibrated_alpha", "alpha_shift_bp"]])

nh.plot_lgm_calibration_summary(configured_lgm, calibrated_lgm, title="Fresh ORE calibration: configured template vs calibrated output")

lgm_ccys = ["EUR", "USD", "GBP", "CHF", "JPY"]
lgm_ccy_rows = []
lgm_alpha_curves = []
lgm_kappa_curves = []

for ccy in lgm_ccys:
    cfg = parse_lgm_params_from_simulation_xml(calibration_meta["simulation_xml"], ccy_key=ccy)
    cal = parse_lgm_params_from_calibration_xml(calibration_meta["calibration_xml"], ccy_key=ccy)
    alpha_cfg_ccy = nh.lgm_param_frame(cfg, param="alpha")
    alpha_cal_ccy = nh.lgm_param_frame(cal, param="alpha")
    kappa_cfg_ccy = nh.lgm_param_frame(cfg, param="kappa")
    kappa_cal_ccy = nh.lgm_param_frame(cal, param="kappa")

    lgm_ccy_rows.append(
        {
            "ccy": ccy,
            "alpha_bucket_count": len(alpha_cal_ccy),
            "kappa_bucket_count": len(kappa_cal_ccy),
            "alpha_mean_shift_bp": 1.0e4 * (alpha_cal_ccy["value"].mean() - alpha_cfg_ccy["value"].mean()),
            "kappa_mean_shift_bp": 1.0e4 * (kappa_cal_ccy["value"].mean() - kappa_cfg_ccy["value"].mean()),
        }
    )

    lgm_alpha_curves.append(
        alpha_cal_ccy.assign(ccy=ccy).rename(columns={"value": "calibrated_alpha"})
    )
    lgm_kappa_curves.append(
        kappa_cal_ccy.assign(ccy=ccy).rename(columns={"value": "calibrated_kappa"})
    )

lgm_ccy_summary = pd.DataFrame(lgm_ccy_rows).sort_values("ccy")
lgm_alpha_curve_df = pd.concat(lgm_alpha_curves, ignore_index=True)
lgm_kappa_curve_df = pd.concat(lgm_kappa_curves, ignore_index=True)

display(lgm_ccy_summary)
display(lgm_alpha_curve_df.head(12))
display(lgm_kappa_curve_df.head(12))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharex=False)
for ccy, grp in lgm_alpha_curve_df.groupby("ccy"):
    g = grp.sort_values("horizon_years")
    axes[0].step(g["horizon_years"], g["calibrated_alpha"], where="post", linewidth=1.8, label=ccy)
axes[0].set_title("Calibrated alpha by currency")
axes[0].set_xlabel("Horizon (years)")
axes[0].set_ylabel("Alpha")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

for ccy, grp in lgm_kappa_curve_df.groupby("ccy"):
    g = grp.sort_values("horizon_years")
    axes[1].step(g["horizon_years"], g["calibrated_kappa"], where="post", linewidth=1.8, label=ccy)
axes[1].set_title("Calibrated kappa by currency")
axes[1].set_xlabel("Horizon (years)")
axes[1].set_ylabel("Kappa")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()
plt.close(fig)


## External Python-vs-ORE calibration parity

The notebook's fresh ORE calibration run keeps the original `Hagan` volatility parametrization, while the current
external Python calibration backend is implemented for the `HullWhite/HullWhite` subset. So this section derives a
`HullWhite` variant of the same ORE example, runs it fresh, and then compares the external Python calibration
against that derived case.

This section runs the external benchmark on the fresh case and summarizes three things:
- whether the basket expiries line up with ORE
- whether the helper market values are matched under the calibrated Python result
- how close the calibrated sigma buckets are to ORE for representative currencies


In [ ]:
calibration_case_root = Path(calibration_hw_parity_meta["fresh_output_dir"]).parent
parity_reports = {}
for ccy in ["EUR", "USD"]:
    out_json = calibration_case_root / f"python_parity_report_{ccy.lower()}.json"
    cmd = [
        sys.executable,
        str(CALIBRATION_PARITY_SCRIPT),
        "--case-root",
        str(calibration_case_root),
        "--currency",
        ccy,
        "--out-json",
        str(out_json),
    ]
    completed = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if not out_json.exists():
        raise FileNotFoundError(f"Parity report not written for {ccy}: {out_json}")
    parity_reports[ccy] = json.loads(out_json.read_text())

parity_summary_rows = [
    {
        "ccy": ccy,
        "status": "ok",
        "vol_source": report["vol_source"],
        "vol_type": report["vol_type"],
        "basket_size": report["basket_size"],
        "basket_grid_match": report["basket_expiry_time_matches_ore_grid"],
        "python_valid": report["python_valid"],
        "python_rmse": report["python_rmse"],
        "max_abs_sigma_diff": report["max_abs_diff"],
        "max_rel_sigma_diff": report["max_rel_diff"],
    }
    for ccy, report in sorted(parity_reports.items())
]
parity_summary = pd.DataFrame(parity_summary_rows)
display(parity_summary)

parity_point_rows = []
for ccy, report in sorted(parity_reports.items()):
    first_point = report["points"][0]
    last_point = report["points"][-1]
    parity_point_rows.extend(
        [
            {
                "ccy": ccy,
                "point": "first",
                "expiry": first_point["expiry"],
                "term": first_point["term"],
                "market_vol": first_point["market_vol"],
                "market_value": first_point["market_value"],
                "model_value": first_point["model_value"],
                "model_vol": first_point["model_vol"],
            },
            {
                "ccy": ccy,
                "point": "last",
                "expiry": last_point["expiry"],
                "term": last_point["term"],
                "market_vol": last_point["market_vol"],
                "market_value": last_point["market_value"],
                "model_value": last_point["model_value"],
                "model_vol": last_point["model_vol"],
            },
        ]
    )
display(pd.DataFrame(parity_point_rows))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharex=False)
for ccy, report in sorted(parity_reports.items()):
    x = np.arange(len(report["ore_sigmas"]))
    axes[0].plot(x, report["ore_sigmas"], linewidth=1.8, label=f"{ccy} ORE")
    axes[0].plot(x, report["python_sigmas"], linewidth=1.4, linestyle="--", label=f"{ccy} Python")
axes[0].set_title("ORE vs Python calibrated sigma buckets")
axes[0].set_xlabel("Bucket index")
axes[0].set_ylabel("Sigma")
axes[0].grid(True, alpha=0.3)
axes[0].legend(ncol=2)

for ccy, report in sorted(parity_reports.items()):
    x = np.arange(len(report["ore_sigmas"]))
    diffs_bp = 1.0e4 * (np.array(report["python_sigmas"]) - np.array(report["ore_sigmas"]))
    axes[1].plot(x, diffs_bp, linewidth=1.8, marker="o", label=ccy)
axes[1].set_title("Python minus ORE sigma difference (bp)")
axes[1].set_xlabel("Bucket index")
axes[1].set_ylabel("Sigma diff x 1e4")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()
plt.close(fig)


The important part of this benchmark is not just the final sigma difference. The benchmark now uses ORE's emitted
`marketdata.csv` and `todaysmarketcalibration.csv` to rebuild the calibration surface and the discount/forward
curves, so a small parameter difference is interpretable. Earlier versions that used synthetic flat helper vols
produced misleading failures. In notebook 3 this compare is run on a derived `HullWhite` variant of the ORE case
so the external Python backend and the ORE calibration are actually solving the same supported problem.


The key visual is the alpha comparison. The template starts from a flat `1%` volatility line, while the calibrated
output bends upward across maturities. Kappa stays flat because this case calibrates volatility but not reversion.
The additional multi-currency tables make it clear which currencies carry their own calibrated alpha/kappa term
structures in the same run.


## Key takeaways

- Curve fitting and LGM parameter extraction belong in the same story because they come from the same run setup.
- The fitted grid is much denser than the market pillars, so visualizing both levels prevents false precision.
- The direct extractor path and the native session bridge now explicitly show one shared curve-extraction contract.
- The same fitter stack also works without XML, which is useful for small programmatic demos and tests.
- The notebook now shows a clean Python curve fitter demo on one USD curve with deposits, FRAs, and swaps as inputs.
- The Python fitter section now shows discount factors, zero rates, and instantaneous forwards side by side.
- The notebook now shows a real fresh ORE calibration run, not a flat parameter template masquerading as calibrated output.
- The calibration section now includes the other configured currencies, not only EUR.
- In the chosen case, `alpha` is calibrated while `kappa` remains fixed, so that asymmetry is expected.
- `load_from_ore_xml(...)` remains the clean bridge from ORE artifacts to Python-side model inputs.
- The new external calibration parity check is only credible because it now rebuilds helper inputs from ORE output artifacts, not from synthetic placeholder vols.
- The parity compare in this notebook is run on a derived HullWhite variant of the calibration case so it reflects an actual supported external-vs-ORE calibration comparison.

## Where this connects next

The next notebook uses the Python LGM stack directly for a transparent IRS exposure and XVA walkthrough.
